In [2]:
# making a function to append a value to a list 
def append_val(val, lst=None) :
    if lst ==None :
        lst = []
    lst.append(val)

    return lst 

In [3]:
append_val(45)

[45]

In [4]:
append_val(90)

[90]

In [7]:
# impliment a bsiac queue... 
class TaskQueue:
    def __init__(self):
        self._tasks = [] # list of dicts 

    def add_task(self,task_id: str,priority: int, payload: dict) -> None:
        self._tasks.append({ "id": task_id,
                             "priority": priority,
                             "payload": payload,
                             "status": "pending"})

    def get_pending(self)-> list:
        return [t for t in self._tasks if t["status"] == "pending"]

    def get_by_id(self, task_id: str)-> dict:

        for t in self._tasks:
            if t["id"]== task_id:

                return t

        return None 
    

In [9]:
# 1. Instantiate the queue
queue = TaskQueue()

# 2. Add some tasks using your add_task method
# Arguments: task_id, priority, payload (a dict)
queue.add_task("task_001", 10, {"action": "render_video", "frame": 45})
queue.add_task("task_002", 5, {"action": "send_email", "to": "user@example.com"})

print("--- Testing get_pending() ---")
# 3. Get all pending tasks
pending_tasks = queue.get_pending()
print(f"Found {len(pending_tasks)} pending tasks:")
for task in pending_tasks:
    print(task)

print("\n--- Testing get_by_id() ---")
# 4. Find a specific task by its ID
target_id = "task_002"
found_task = queue.get_by_id(target_id)
print(f"Result for ID '{target_id}':", found_task)

--- Testing get_pending() ---
Found 2 pending tasks:
{'id': 'task_001', 'priority': 10, 'payload': {'action': 'render_video', 'frame': 45}, 'status': 'pending'}
{'id': 'task_002', 'priority': 5, 'payload': {'action': 'send_email', 'to': 'user@example.com'}, 'status': 'pending'}

--- Testing get_by_id() ---
Result for ID 'task_002': {'id': 'task_002', 'priority': 5, 'payload': {'action': 'send_email', 'to': 'user@example.com'}, 'status': 'pending'}


### leve 2 Priority Processing 

In [11]:
def pop_higghest_priority(self)-> dict : 
    pending= [t for t in self_tasks if t["status"] == "pending"]

    if not pending:
        return None 
    task =max(pending, key =lambda t : t["priority"])
    task["status"] = "in_progress"
    return task 
def complete_task(self,task_id: str, result: any) -> None:

    task = self.get_by_id(task_id)
    if task :
        task["status"] = "done"
        task["result"] = result 

def get_stats(self) -> dict:
    from collections import Counter
    counts = Counter(t["status"] for t in self._tasks)
    return dict(counts)



## Level 3 Async Handler 

In [13]:
import asyncio
from asyncio import Queue ## imp

class AsyncTaskQueue:
    def __init__(self, num_workers: int = 3):
        self._q = Queue()
        self._results = {}
        self._lock = asyncio.Lock()
        self._num_workers = num_workers

    async def add_task(self, task_id: str, payload: dict):
        await self._q.put({"id": task_id, "payload": payload})

    async def _worker(self, worker_id: int, handler):
        while True:
            task = await self._q.get()
            if task is None:
                break
            result = await handler(task["payload"])
            async with self._lock:
                self._results[task["id"]] = result
            self._q.task_done()

    async def run(self, tasks: list, handler):
        for task in tasks:
            await self._q.put(task)
        workers = [asyncio.create_task(self._worker(i, handler))
                   for i in range(self._num_workers)]
        await self._q.join()
        for _ in workers:
            await self._q.put(None)
        await asyncio.gather(*workers)
        return self._results

### Practice Task 1: Task Queue System

**Spec:** Implement a task queue where tasks have IDs, priorities, and can be processed by worker threads.

**Level 1 — Basic Queue**

```python
class TaskQueue:
    def __init__(self):
        self._tasks = []  # list of dicts

    def add_task(self, task_id: str, priority: int, payload: dict) -> None:
        self._tasks.append({
            "id": task_id, "priority": priority,
            "payload": payload, "status": "pending"
        })

    def get_pending(self) -> list:
        return [t for t in self._tasks if t["status"] == "pending"]

    def get_by_id(self, task_id: str) -> dict:
        for t in self._tasks:
            if t["id"] == task_id:
                return t
        return None
```

**Level 2 — Priority Processing**

```python
    def pop_highest_priority(self) -> dict:
        pending = [t for t in self._tasks if t["status"] == "pending"]
        if not pending:
            return None
        task = max(pending, key=lambda t: t["priority"])
        task["status"] = "in_progress"
        return task

    def complete_task(self, task_id: str, result: any) -> None:
        task = self.get_by_id(task_id)
        if task:
            task["status"] = "done"
            task["result"] = result

    def get_stats(self) -> dict:
        from collections import Counter
        counts = Counter(t["status"] for t in self._tasks)
        return dict(counts)
```

**Level 3 — Concurrent Workers (asyncio)**

```python
import asyncio
from asyncio import Queue

class AsyncTaskQueue:
    def __init__(self, num_workers: int = 3):
        self._q = Queue()
        self._results = {}
        self._lock = asyncio.Lock()
        self._num_workers = num_workers

    async def add_task(self, task_id: str, payload: dict):
        await self._q.put({"id": task_id, "payload": payload})

    async def _worker(self, worker_id: int, handler):
        while True:
            task = await self._q.get()
            if task is None:
                break
            result = await handler(task["payload"])
            async with self._lock:
                self._results[task["id"]] = result
            self._q.task_done()

    async def run(self, tasks: list, handler):
        for task in tasks:
            await self._q.put(task)
        workers = [asyncio.create_task(self._worker(i, handler))
                   for i in range(self._num_workers)]
        await self._q.join()
        for _ in workers:
            await self._q.put(None)
        await asyncio.gather(*workers)
        return self._results
```

**Level 4 — Rate Limiting**

```python
    async def _rate_limited_worker(self, sem: asyncio.Semaphore, handler):
        while True:
            task = await self._q.get()
            if task is None:
                break
            async with sem:                    # max N concurrent actual executions
                result = await handler(task["payload"])
            async with self._lock:
                self._results[task["id"]] = result
            self._q.task_done()
```

**Level 5 — Retries and Error Handling**

```python
    async def _worker_with_retry(self, handler, max_retries: int = 3):
        while True:
            task = await self._q.get()
            if task is None:
                break
            for attempt in range(max_retries):
                try:
                    result = await handler(task["payload"])
                    async with self._lock:
                        self._results[task["id"]] = {"status": "ok", "result": result}
                    break
                except Exception as e:
                    if attempt == max_retries - 1:
                        async with self._lock:
                            self._results[task["id"]] = {"status": "error", "error": str(e)}
            self._q.task_done()
```

**Level 6 — Task Dependencies (DAG)**

```python
    async def run_dag(self, tasks: dict, handler):
        # tasks = {"A": {"deps": [], "payload": ...}, "B": {"deps": ["A"], ...}}
        completed = set()
        results = {}
        lock = asyncio.Lock()

        async def run_task(name):
            # Wait for all deps
            while True:
                async with lock:
                    if all(d in completed for d in tasks[name]["deps"]):
                        break
                await asyncio.sleep(0.01)
            result = await handler(tasks[name]["payload"])
            async with lock:
                results[name] = result
                completed.add(name)

        await asyncio.gather(*[run_task(name) for name in tasks])
        return results